# Noise comparison
This notebook generates simulated data under different noise families. It then computes the power measure according to the different alternative hypothesis specification on the Andreoni and Miller constraints. Finally, it compares the power measures under these different noise specifications.

## 1. Define qual objects across simulations

In [4]:
from power_rptests.data_generation import *
from power_rptests.data_plotting import *
import pickle
import openpyxl

path = "/Users/federicobassi/Desktop/power_rptests"
simulated_data = path + "/simulated_data"
figures = path + "/figures/"

In [5]:
# Constants across simulations
utility_function = "ces"
param_distribution = { "ces": {"alpha": lambda rng: rng.uniform(0.5, 1),
                                "rho":   lambda rng: rng.uniform(-0.5, 0.95),}}

andreoni_miller_budgets = [(120, 40), (40, 120), (120, 60), (60, 120), (150, 75),
                               (75, 150), (60, 60), (100, 100), (80, 80), (160, 40), (40, 160)]
andreoni_miller_budgets = [(10*ms, 10*mo) for ms, mo in andreoni_miller_budgets]
andreoni_miller_budgets.sort(key=lambda x: x[0])
andreoni_miller_budgets = {"budgets": andreoni_miller_budgets, "label":"Andreoni-Miller budgets"}

with open(simulated_data+"/budget_constraints_alg2.pkl", "rb") as f:
    optimized_budgets = pickle.load(f)
optimized_budgets.sort(key=lambda x: x[0])
optimized_budgets = {"budgets": optimized_budgets, "label":"Optimised budgets"}

## 2. Generate a dataframe with different noise specifications

In [9]:
noise_models = ["jittering","lapses","misperception","quantal_response"]

dfs = []

for noise in noise_models:
    print(f"Working on noise type: {noise}")
    if noise == "jittering":
        df = simulate(
            andreoni_miller_budgets,
            "ces",
            param_distribution,
            noise_type=noise,
            std=[50,100, 150],
            free_disposal=True,
            n_samples=1000,
            seed=123
        )

    elif noise == "lapses":
        df = simulate(
            andreoni_miller_budgets,
            "ces",
            param_distribution,
            noise_type=noise,
            lapse_prob=[0.05,0.1,0.2],
            n_samples=1000,
            seed=123
        )

    elif noise == "misperception":
        df = simulate(
            andreoni_miller_budgets,
            "ces",
            param_distribution,
            noise_type=noise,
            price_noise_sd=[0.01,0.05,0.1],
            n_samples=1000,
            seed=123
        )

    elif noise == "quantal_response":
        df = simulate(
            andreoni_miller_budgets,
            "ces",
            param_distribution,
            noise_type=noise,
            lambda_qr=[0.1,0.5,1],
            grid_size=51,
            n_samples=1000,
            seed=123
        )

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

Working on noise type: jittering
Working on noise type: lapses
Working on noise type: misperception
Working on noise type: quantal_response


/Users/federicobassi/Desktop/power_rptests/src/power_rptests/data_generation.py:26: RuntimeWarning: divide by zero encountered in scalar power
  return (alpha * (x ** rho) + (1.0 - alpha) * (y ** rho)) ** (1.0 / rho)


In [8]:
df_all

,id,x_intercept,y_intercept,utility,budget_label,alpha,rho,opt_x,opt_y,noisy_x,noisy_y,noise_type,noise_param,noise_free_disposal,noise_grid_size
0,1,400.0,1200.0,ces,Andreoni-Miller budgets,0.841176,0.310912,349.009432,152.971704,328.857373,102.171204,jittering,50.0,True,NaN
1,1,400.0,1600.0,ces,Andreoni-Miller budgets,0.841176,0.310912,342.950068,228.199727,260.026948,228.258938,jittering,50.0,True,NaN
2,1,600.0,1200.0,ces,Andreoni-Miller budgets,0.841176,0.310912,534.914289,130.171421,494.811458,135.587457,jittering,50.0,True,NaN
3,1,600.0,600.0,ces,Andreoni-Miller budgets,0.841176,0.310912,550.965363,49.034637,508.978186,46.653710,jittering,50.0,True,NaN
4,1,750.0,1500.0,ces,Andreoni-Miller budgets,0.841176,0.310912,668.642862,162.714276,705.713593,84.987101,jittering,50.0,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13195,100,1000.0,1000.0,ces,Andreoni-Miller budgets,0.515147,-0.109444,513.653866,486.346134,500.000000,500.000000,quantal_response,1.0,NaN,51.0
13196,100,1200.0,400.0,ces,Andreoni-Miller budgets,0.515147,-0.109444,583.879934,205.373355,576.000000,208.000000,quantal_response,1.0,NaN,51.0
13197,100,1200.0,600.0,ces,Andreoni-Miller budgets,0.515147,-0.109444,595.875574,302.062213,648.000000,276.000000,quantal_response,1.0,NaN,51.0
13198,100,1500.0,750.0,ces,Andreoni-Miller budgets,0.515147,-0.109444,744.844468,377.577766,720.000000,390.000000,quantal_response,1.0,NaN,51.0
